## 연습 1: 원시 예약 데이터 수집 (Bronze 계층)

**Bronze 계층**은 소스 시스템에서 도착한 그대로의 원시, 수정되지 않은 데이터를 저장합니다. 프로덕션 파이프라인에서는 데이터가 부동산 관리 시스템(PMS)에서 도착합니다. 이 랩에서는 5개의 GlobStay 숙소에 걸친 예약을 나타내는 샘플 데이터셋이 제공됩니다.

데이터 문제는 의도적입니다 — 이는 Silver 계층에서 해결할 실제 상황을 반영합니다.

### ✅ 제공됨: Unity Catalog 구조 및 샘플 데이터 생성

아래 두 셀을 실행하여 다음을 수행하세요:
- `bronze`, `silver`, `gold` 스키마를 가진 `hospitality_lab` 카탈로그 생성
- 의도적인 품질 문제를 포함하는 샘플 원시 예약 DataFrame 생성

출력을 주의 깊게 살펴보세요 — 연습 2를 다루기 전에 데이터 문제를 이해해야 합니다.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

In [ ]:
# ✅ 이 셀을 실행하세요 — Unity Catalog 구조를 생성합니다

catalog = "hospitality_lab"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog} MANAGED LOCATION '{storage_root}' COMMENT 'GlobStay 호스피탈리티 분석 플랫폼'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze COMMENT '원시 수집된 예약 데이터'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver COMMENT '정제 및 검증된 예약 데이터'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold   COMMENT '분석 준비된 집계 데이터'")

print(f"✅ 카탈로그 '{catalog}' 및 스키마 bronze / silver / gold 가 생성되었습니다.")

In [ ]:
# ✅ 이 셀을 실행하세요 — 원시 예약 DataFrame을 생성합니다

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("booking_id",      StringType(),  True),
    StructField("property_id",     StringType(),  True),
    StructField("property_name",   StringType(),  True),
    StructField("check_in_date",   StringType(),  True),
    StructField("check_out_date",  StringType(),  True),
    StructField("room_type",       StringType(),  True),
    StructField("rate_per_night",  DoubleType(),  True),
    StructField("guest_id",        StringType(),  True),
    StructField("guest_name",      StringType(),  True),
    StructField("guest_email",     StringType(),  True),
    StructField("booking_status",  StringType(),  True),
    StructField("channel",         StringType(),  True),
    StructField("adults",          IntegerType(), True),
    StructField("children",        IntegerType(), True),
])

raw_data = [
    # (booking_id, property_id, property_name, check_in, check_out, room_type, rate, guest_id, guest_name, guest_email, status, channel, adults, children)
    ("B001","P01","Grand Harbor Hotel",   "2025-06-15","2025-06-18","Deluxe",  250.00,"G001","Alex Rivera",      "alex.r@email.com",    "confirmed","direct",    2,0),
    ("B002","P01","Grand Harbor Hotel",   "2025-06-20","2025-06-22","Suite",   450.00,"G002","Maria Santos",     "m.santos@corp.com",   "confirmed","corporate", 1,0),
    ("B003","P02","Sunset Beach Resort",  "2025-07-01","2025-07-08","Sea View",180.00,"G003","James Chen",       "jchen@email.com",     "confirmed","ota",       2,2),
    ("B004","P02","Sunset Beach Resort",  "2025-07-10","2025-07-12","Garden",  140.00,"G004","Sarah Johnson",    None,                  "confirmed","direct",    2,0),  # null 이메일
    ("B005","P03","Mountain Pines Lodge", "2025-08-05","2025-08-10",None,       120.00,"G005","Robert Kim",       "rkim@email.com",      "cancelled","ota",       1,0),  # null room_type
    ("B006","P03","Mountain Pines Lodge", "2025-08-15","2025-08-12","Standard",110.00,"G006","Emily Brown",      "ebrown@email.com",    "confirmed","direct",    2,1),  # check_out < check_in
    ("B001","P01","Grand Harbor Hotel",   "2025-06-15","2025-06-18","Deluxe",  250.00,"G001","Alex Rivera",      "alex.r@email.com",    "confirmed","direct",    2,0),  # B001의 중복
    ("B007","P04","City Center Inn",      "2025-09-10","2025-09-11","Standard",-50.00,"G007","Michael Turner",   "mturner@email.com",   "confirmed","corporate", 1,0),  # 음수 요금
    ("B008","P04","City Center Inn",      "2025-09-15","2025-09-18","Superior",195.00,"G008","Lisa Wong",        "lwong@email.com",     "confirmed","ota",       2,0),
    ("B009","P01","Grand Harbor Hotel",   "2025-10-01","2025-10-05","Suite",   480.00,"G009","Carlos Mendez",    "cmendes@biz.com",     "confirmed","corporate", 2,0),
    ("B010","P02","Sunset Beach Resort",  "2025-11-20","2025-11-25","Sea View",155.00,"G010","Aisha Patel",      "aisha.p@email.com",   "confirmed","direct",    2,1),
    ("B011","P03","Mountain Pines Lodge", "2025-12-22","2025-12-26","Deluxe",  180.00,"G011","Thomas Muller",    "tmuller@email.com",   "confirmed","ota",       2,0),
    ("B012","P04","City Center Inn",      "2025-10-08","2025-10-09","Standard",  0.00,"G012","Yuki Tanaka",      "ytanaka@email.com",   "confirmed","direct",    1,0),  # 0 요금
    ("B013","P01","Grand Harbor Hotel",   "2025-07-04","2025-07-07","Deluxe",  275.00,"G013","Sophie Anderson",  "sanderson@travel.com","confirmed","ota",       2,2),
    ("B014","P02","Sunset Beach Resort",  "2025-08-20","2025-08-27","Suite",   350.00,"G014","Daniel Hart",      "dhart@email.com",     "pending",  "direct",    2,0),
    ("B015","P05","Airport Transit Hotel","2025-06-30","2025-07-01","Standard", 99.00,"G015","Priya Sharma",     None,                  "confirmed","ota",       1,0),  # null 이메일
]

raw_df = spark.createDataFrame(raw_data, schema)
raw_df.display()
print(f"\n원시 레코드: {raw_df.count()}")

### 🎯 챌린지: Bronze 테이블에 원시 데이터 쓰기

`raw_df`를 `hospitality_lab.bronze` 스키마의 **`bookings_bronze`**라는 Delta 테이블에 쓰세요.

**요구 사항:**
- 셀을 다시 실행할 수 있도록 `overwrite` 모드 사용
- 쓰기 후, 레코드 수가 원시 DataFrame과 일치하는지 확인

> 🤖 **Genie Code 팁:** *"How do I write a PySpark DataFrame to a Unity Catalog table using saveAsTable with overwrite mode?"* 라고 물어보세요.

In [ ]:
raw_df.write.mode("overwrite").saveAsTable("hospitality_lab.bronze.bookings_bronze")

count = spark.table("hospitality_lab.bronze.bookings_bronze").count()
print(f"✅ Bronze 테이블 생성됨 — {count}개 레코드")
spark.table("hospitality_lab.bronze.bookings_bronze").display()

## 연습 2: 예약 데이터 정제 및 검증 (Silver 계층)

**Silver 계층**은 다운스트림 프로세스가 신뢰할 수 있는 정제 및 검증된 데이터를 포함합니다. bronze 계층에 존재하는 각 품질 문제를 해결해야 합니다:

| 문제 | 발견 위치 | 규칙 |
|---|---|---|
| 중복 예약 | B001이 두 번 나타남 | `booking_id`별 첫 번째 항목만 유지 |
| Null `room_type` | B005 | room type이 없는 레코드 제외 |
| 잘못된 날짜 순서 | B006 (`check_out`이 `check_in`보다 앞섬) | `check_out_date <= check_in_date`인 레코드 제외 |
| 양수가 아닌 요금 | B007 (–50), B012 (0) | `rate_per_night <= 0`인 레코드 제외 |

> **참고:** Null `guest_email` 값(B004, B015)은 허용됩니다 — 모든 예약에 이메일 주소가 필요한 것은 아닙니다.

### 🎯 챌린지: Silver 테이블 생성

`hospitality_lab.bronze.bookings_bronze`에서 읽고, 네 가지 정제 규칙을 모두 적용한 다음, 결과를 **`hospitality_lab.silver.bookings_silver`**(overwrite 모드)에 쓰세요.

**힌트:**
- 중복 제거에는 `.dropDuplicates(["booking_id"])` 사용
- null 제거에는 `.filter(col("room_type").isNotNull())` 사용
- 날짜 비교를 위해 `pyspark.sql.functions`에서 `to_date`와 `datediff` 임포트
- 각 규칙에 대해 `.filter()` 호출 연결

쓰기 후, Bronze와 Silver 레코드 수를 모두 출력하여 레코드가 제거되었는지 확인하세요.

> 🤖 **Genie Code 팁:** *"How do I filter PySpark rows where check_out_date is not after check_in_date using date comparison?"* 라고 물어보세요.

In [ ]:
from pyspark.sql.functions import col, to_date, datediff

silver_df = (
    spark.table("hospitality_lab.bronze.bookings_bronze")
    .dropDuplicates(["booking_id"])
    .filter(col("room_type").isNotNull())
    .withColumn("_cin",  to_date(col("check_in_date")))
    .withColumn("_cout", to_date(col("check_out_date")))
    .filter(datediff(col("_cout"), col("_cin")) > 0)
    .filter(col("rate_per_night") > 0)
    .drop("_cin", "_cout")
)

silver_df.write.mode("overwrite").saveAsTable("hospitality_lab.silver.bookings_silver")

bronze_count = spark.table("hospitality_lab.bronze.bookings_bronze").count()
silver_count = spark.table("hospitality_lab.silver.bookings_silver").count()
print(f"Bronze 레코드 : {bronze_count}")
print(f"Silver 레코드 : {silver_count}  ({bronze_count - silver_count}개 제거됨)")
print("✅ Silver 테이블이 성공적으로 생성되었습니다")
spark.table("hospitality_lab.silver.bookings_silver").display()

## 연습 3: 분석 준비 데이터 구축 (Gold 계층)

**Gold 계층**은 고도로 큐레이션된 비즈니스 대면 데이터셋을 포함합니다. GlobStay 경영진은 두 개의 분석 테이블을 원합니다:

1. **`property_performance`** — 호텔 숙소별 총 확정 예약 수, 판매된 객실 박수, 매출
2. **`channel_performance`** — 예약 채널(direct, OTA, corporate)별로 구분된 예약량 및 매출

### 🎯 챌린지 3a: 숙소 성과 테이블

Spark SQL을 사용하여 Gold 테이블 `hospitality_lab.gold.property_performance`를 생성하세요.

**요구 사항:**
- 소스: `hospitality_lab.silver.bookings_silver`
- `booking_status = 'confirmed'`만 필터링
- 열: `property_id`, `property_name`, `confirmed_bookings`, `total_nights`, `total_revenue`
- `total_nights`는 `DATEDIFF(check_out_date, check_in_date)`로 계산
- `total_revenue`는 `total_nights × rate_per_night`로 계산
- `total_revenue DESC`로 정렬

> 🤖 **Genie Code 팁:** *"Write a Spark SQL CREATE OR REPLACE TABLE statement that aggregates booking revenue per property using DATEDIFF and SUM."* 라고 물어보세요.

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE hospitality_lab.gold.property_performance AS
SELECT
    property_id,
    property_name,
    COUNT(*)                                                           AS confirmed_bookings,
    SUM(DATEDIFF(check_out_date, check_in_date))                       AS total_nights,
    SUM(DATEDIFF(check_out_date, check_in_date) * rate_per_night)      AS total_revenue
FROM hospitality_lab.silver.bookings_silver
WHERE booking_status = 'confirmed'
GROUP BY property_id, property_name
ORDER BY total_revenue DESC
""")

print("✅ property_performance 테이블이 생성되었습니다")
spark.table("hospitality_lab.gold.property_performance").display()

### 🎯 챌린지 3b: 예약 채널 성과 테이블

각 예약 채널이 GlobStay 매출에 어떻게 기여하는지 보여주는 Gold 테이블 `hospitality_lab.gold.channel_performance`를 생성하세요.

**요구 사항:**
- 소스: `hospitality_lab.silver.bookings_silver`
- `booking_status = 'confirmed'`만 필터링
- 열: `channel`, `booking_count`, `total_revenue`, `avg_revenue_per_booking`(소수점 2자리로 반올림)
- `total_revenue DESC`로 정렬

> 🤖 **Genie Code 팁:** *"How do I calculate average revenue per booking in Spark SQL using ROUND and AVG?"* 라고 물어보세요.

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE hospitality_lab.gold.channel_performance AS
SELECT
    channel,
    COUNT(*)                                                                        AS booking_count,
    SUM(DATEDIFF(check_out_date, check_in_date) * rate_per_night)                   AS total_revenue,
    ROUND(AVG(DATEDIFF(check_out_date, check_in_date) * rate_per_night), 2)         AS avg_revenue_per_booking
FROM hospitality_lab.silver.bookings_silver
WHERE booking_status = 'confirmed'
GROUP BY channel
ORDER BY total_revenue DESC
""")

print("✅ channel_performance 테이블이 생성되었습니다")
spark.table("hospitality_lab.gold.channel_performance").display()

## 연습 4: 파이프라인 노트북의 오류 처리

노트북이 Lakeflow Job 작업으로 실행될 때는 오류를 우아하게 처리하고 그 결과를 다운스트림 작업에 전달해야 합니다. 두 가지 핵심 패턴은 다음과 같습니다:

1. **`PySparkException`을 사용한 `try/except`** — 특정 Spark 오류를 포착하고 노트북을 중단시키지 않고 처리
2. **`dbutils.notebook.exit()`** — 작업이 검사할 수 있는 구조화된 메시지로 성공 또는 실패를 신호

### 🎯 챌린지 4a: try/except로 안전한 테이블 읽기 구현

다음을 수행하는 함수 `safe_read_table(table_name)`을 작성하세요:
1. `spark.table(table_name)`을 사용하여 테이블 읽기 시도
2. `pyspark.errors`에서 `PySparkException` 포착
3. 테이블을 찾을 수 없는 경우 오류 클래스와 함께 설명 메시지 출력
4. 오류 시 `None`, 성공 시 DataFrame 반환

유효한 테이블과 존재하지 않는 테이블로 테스트하여 두 경로가 모두 작동하는지 확인하세요.

**힌트:**
- 임포트: `from pyspark.errors import PySparkException`
- 구조화된 오류 코드를 얻으려면 `e.getErrorClass()` 사용
- 처리하지 않은 예기치 않은 오류는 다시 발생시키기

> 🤖 **Genie Code 팁:** *"Show me how to catch a PySparkException in Python and print its error class."* 라고 물어보세요.

In [ ]:
from pyspark.errors import PySparkException

def safe_read_table(table_name):
    try:
        df = spark.table(table_name)
        count = df.count()
        print(f"✅ '{table_name}' 읽음 — {count}개 레코드")
        return df
    except PySparkException as e:
        if e.getErrorClass() in ("TABLE_OR_VIEW_NOT_FOUND", "SCHEMA_NOT_FOUND"):
            print(f"❌ 테이블을 찾을 수 없음: '{table_name}' ({e.getErrorClass()})")
            return None
        raise  # 예기치 않은 오류는 다시 발생

# 테스트: 유효한 테이블
df_valid = safe_read_table("hospitality_lab.silver.bookings_silver")

# 테스트: 존재하지 않는 테이블
df_missing = safe_read_table("hospitality_lab.silver.nonexistent_table")

### 🎯 챌린지 4b: dbutils.notebook.exit()로 작업 결과 신호

노트북이 Lakeflow Job 작업으로 실행될 때, `dbutils.notebook.exit()`를 호출하면 다운스트림 작업이 `{{tasks.<task_name>.values.<key>}}`를 사용하여 결과를 읽을 수 있습니다.

다음을 수행하는 함수 `report_pipeline_outcome(table_name)`을 작성하세요:
1. 지정된 테이블을 읽고 레코드 수를 셈
2. 성공 시, `"SUCCESS: Processed <n> records from <table_name>"` 메시지로 `dbutils.notebook.exit()` 호출
3. 모든 예외 발생 시, `"FAILED: <error message>"` 메시지로 `dbutils.notebook.exit()` 호출

> ⚠️ **경고:** `dbutils.notebook.exit()`는 노트북 실행을 즉시 중단합니다. 함수를 정의하되 **호출하지 마세요** — 대신 호출을 주석 처리된 상태로 두세요. 작업에서는 이를 호출하는 것이 올바른 동작입니다.

> 🤖 **Genie Code 팁:** *"How does dbutils.notebook.exit work in Azure Databricks when a notebook runs as a job task?"* 라고 물어보세요.

In [ ]:
def report_pipeline_outcome(table_name):
    try:
        count = spark.table(table_name).count()
        dbutils.notebook.exit(f"SUCCESS: Processed {count} records from {table_name}")
    except Exception as e:
        dbutils.notebook.exit(f"FAILED: {str(e)}")

# 테스트하려면 아래 줄의 주석을 해제하세요 (노트북 실행을 중단합니다)
# report_pipeline_outcome("hospitality_lab.silver.bookings_silver")

print("함수가 정의되었습니다. 작업 작업 노트북의 마지막 단계에서만 호출하세요.")

## 연습 5: 매개변수화된 노트북 작업

재사용 가능한 파이프라인 노트북은 런타임에 매개변수를 받습니다 — 코드 변경 없이 동일한 노트북이 다른 시나리오(예: 다른 날짜, 환경 또는 필터 값)에서 작동하도록 만듭니다.

Lakeflow Jobs에서는 두 가지 메커니즘이 사용됩니다:
- **`dbutils.widgets`** — 작업 작업 구성에서 노트북 *안으로* 매개변수 전달
- **`dbutils.jobs.taskValues`** — 동일한 작업 실행 내에서 노트북 작업 *사이에* 값 전달

### 🎯 챌린지 5a: 런타임 위젯 매개변수 추가

기본값이 `confirmed`이고 레이블이 `Booking Status Filter`인 `booking_status`라는 텍스트 위젯을 추가하세요.

그런 다음 위젯 값을 검색하여 `hospitality_lab.silver.bookings_silver`를 필터링하는 데 사용하세요. 필터링된 결과를 표시하고 레코드 수를 출력하세요.

**힌트:**
- `dbutils.widgets.text(name, default, label)`
- `dbutils.widgets.get(name)`은 현재 값을 반환
- Lakeflow Job에서는 작업 **Parameters** 구성이 런타임에 기본값을 재정의

> 🤖 **Genie Code 팁:** *"How do I create a text widget in Databricks and retrieve its value to filter a DataFrame?"* 라고 물어보세요.

In [ ]:
dbutils.widgets.text("booking_status", "confirmed", "Booking Status Filter")
status_filter = dbutils.widgets.get("booking_status")

filtered_df = (
    spark.table("hospitality_lab.silver.bookings_silver")
    .filter(f"booking_status = '{status_filter}'")
)

print(f"상태가 '{status_filter}'인 예약: {filtered_df.count()}")
filtered_df.display()

### 🎯 챌린지 5b: 다운스트림 작업에 값 전달

처리 후, 노트북 작업은 `dbutils.jobs.taskValues.set()`을 사용하여 계산된 값을 다운스트림 작업과 공유할 수 있습니다.

Silver 테이블에서 두 개의 작업 값을 설정하세요:
1. 키 `silver_record_count` — `bookings_silver`의 레코드 수
2. 키 `properties_processed` — 고유한 `property_id` 값의 Python 리스트

그런 다음 설정한 내용을 출력하여 확인할 수 있도록 하세요.

> **참고:** `dbutils.jobs.taskValues`는 Lakeflow Job 실행 내부에서만 사용할 수 있습니다. 대화형 노트북 세션에서는 경고를 출력하고 아무 동작도 하지 않습니다 — 이는 예상된 동작입니다.

> 🤖 **Genie Code 팁:** *"How do I use dbutils.jobs.taskValues.set to pass a list value from one Databricks notebook task to another?"* 라고 물어보세요.

In [ ]:
silver_df = spark.table("hospitality_lab.silver.bookings_silver")
record_count = silver_df.count()
properties = [row.property_id for row in silver_df.select("property_id").distinct().collect()]

dbutils.jobs.taskValues.set(key="silver_record_count",  value=record_count)
dbutils.jobs.taskValues.set(key="properties_processed", value=properties)

print(f"설정된 작업 값:")
print(f"  silver_record_count  = {record_count}")
print(f"  properties_processed = {properties}")